# Collecting the BNF and BNF-EN metadata using BNF Gallica's API

BNF exposes its metadata through a wide range of API formats: intermarcXchange, unimarcXchange and dublin-core.
After exploration of the metadata which can be fetch in each of these formats and comparing it to the existing metadata we have, the **intermarc** format was chosen.

The choice of this format was also motivated by the existance of a very complete [documentation](https://www.google.com/url?q=https://www.bnf.fr/fr/intermarc-bibliographique-de-diffusion%23bnf-zones-fixes&sa=D&source=editors&ust=1718811747871572&usg=AOvVaw2vvRvlAk1791nsBKadBMH-) of each numbered zone, to ensure the collected data was correctly classified.

This notebook thus aims to call the API for the metadata of each of the titles within our corpus which were provided by the BNF.

Once collected, the metadata is stored in a dedicated xml file which should be copied to the [impresso-master-db](https://github.com/impresso/impresso-master-db) repository and uploaded to the MySQL database.

#### Note:
The Gallica API provides a way to query its API for multiple newspaper titles at once. 

However, after experimenting with this approach, it has been identified that the metadata for one title (in this case `jdpl`) was **not present in the response**, even if querying it alone presented no issue.

As a result, this notebook contains both requests, and it adds `jdpl`'s metadata back into the XML with the rest of the title's metadata.

### Imports

In [1]:
import os
import requests
from bs4 import BeautifulSoup
import json
import pymarc

### Code

First defile all the possible API URIs and constants that might be used.

In [2]:
BNFEN_API_MAPPING = {
    "oerennes": "cb32830550k",
    "oecaen": "cb41193642z",
    "lematin": "cb328123058",
    "lepji": "cb32836564q",
    "jdpl": "cb39294634r",
    "legaulois": "cb32779904b",
    "lepetitparisien": "cb34419111x",
}

BNF_API_MAPPING = {
    "excelsior": "cb32771891w",
    "lafronde": "cb327788531",
    "marieclaire": "cb343488519",
    "oeuvre": "cb34429265b",
}

In [19]:
marc_API_URI = "http://catalogue.bnf.fr/api/SRU?version=1.2&operation=searchRetrieve&query=(bib.ark%20any%20%22{query}%22)&recordSchema=intermarcXchange"
master_db_data_dir= '/Users/piconti/impresso/impresso-master-db/impresso_db/data' # TODO adapt to local path
data_dir= '/Users/piconti/impresso/impresso-corpus-metadata/data'  # TODO adapt to local path

## Using the BNS marc21 file as example

The BNS metadata is in Marc21 format, a slight variation compared to the one from BNF.

The metadata for all titles are also in the same file, and the same script will be used to upload BNF and BNS data.
So testing how this data is read inside the script to ensure the BNF metadata is correctly stored is necessary.

In [9]:
bns_marc_file = os.path.join(master_db_data_dir, 'bns-marc21.xml')

In [10]:
bns_marc_array = pymarc.parse_xml_to_array(bns_marc_file)
bns_marc_array[:4]

 None,
 None]

In [12]:
for r in bns_marc_array:
    if r is not None:
        print(f'title: {r["245"]["a"]}')
        print(r.as_dict())

title: Le journal de Genève
{'leader': '02040nas a22005414a 4500', 'fields': [{'001': 'vtls001556959'}, {'003': 'Sz'}, {'005': '20180320142800.0'}, {'006': 'm        m  '}, {'007': 'cr            '}, {'008': '130614d18261991sz dr n s     0    0fre d'}, {'022': {'ind1': ' ', 'ind2': ' ', 'subfields': [{'a': '1010-2108'}]}}, {'024': {'ind1': '7', 'ind2': ' ', 'subfields': [{'a': 'http://permalink.snl.ch/bib/sz001556959'}, {'2': 'permalink'}]}}, {'035': {'ind1': ' ', 'ind2': ' ', 'subfields': [{'a': '(Sz)001556959'}]}}, {'040': {'ind1': ' ', 'ind2': ' ', 'subfields': [{'a': 'Sz'}, {'c': 'Sz'}]}}, {'082': {'ind1': '0', 'ind2': '4', 'subfields': [{'a': '079.49451'}, {'2': '22'}]}}, {'082': {'ind1': '7', 'ind2': '4', 'subfields': [{'a': '050'}, {'2': '22sdnb'}]}}, {'090': {'ind1': ' ', 'ind2': ' ', 'subfields': [{'i': 'dcel'}]}}, {'245': {'ind1': '0', 'ind2': '3', 'subfields': [{'a': 'Le journal de Genève'}]}}, {'246': {'ind1': '1', 'ind2': '3', 'subfields': [{'a': "Journal de Genève des let

## Concatenating the data for all BNF and BNF-EN titles in one file

Concatenate all the ark ids of the BNF titles into 1 for a unique query to the API.

The arks simply need to be concatenated with a space.

In [15]:
ark_base = "ark:/12148"
bnf_en_arks = [f"{ark_base}/{ark_v}" for k, ark_v in BNFEN_API_MAPPING.items()]
bnf_arks = [f"{ark_base}/{ark_v}" for k, ark_v in BNF_API_MAPPING.items()]
bnf_en_arks.extend(bnf_arks)
all_arks = ' '.join(bnf_en_arks)

all_arks

'ark:/12148/cb32830550k ark:/12148/cb41193642z ark:/12148/cb328123058 ark:/12148/cb32836564q ark:/12148/cb39294634r ark:/12148/cb32779904b ark:/12148/cb34419111x ark:/12148/cb32771891w ark:/12148/cb327788531 ark:/12148/cb343488519 ark:/12148/cb34429265b'

Collect the metadata for all titles and save it to an xml file

In [20]:
bnf_marc_file = os.path.join(data_dir, 'bnf-intermarcXchange.xml')

In [46]:
full_bnf_query_uri = marc_API_URI.format(query = all_arks)

resp_all = requests.get(full_bnf_query_uri, timeout=60)
all_contents = BeautifulSoup(resp_all.content, 'xml')

all_contents

<?xml version="1.0" encoding="utf-8"?>
<srw:searchRetrieveResponse xmlns="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:ixm="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:mn="http://catalogue.bnf.fr/namespaces/motsnotices" xmlns:sd="http://www.loc.gov/zing/srw/diagnostic/" xmlns:srw="http://www.loc.gov/zing/srw/">
<srw:version>1.2</srw:version>
<srw:echoedSearchRetrieveRequest>
<srw:version>1.2</srw:version>
<srw:query>(bib.ark any "ark:/12148/cb32830550k ark:/12148/cb41193642z ark:/12148/cb328123058 ark:/12148/cb32836564q ark:/12148/cb39294634r ark:/12148/cb32779904b ark:/12148/cb34419111x ark:/12148/cb32771891w ark:/12148/cb327788531 ark:/12148/cb343488519 ark:/12148/cb34429265b")</srw:query>
</srw:echoedSearchRetrieveRequest>
<srw:numberOfRecords>11</srw:numberOfRecords>
<srw:records>
<srw:record>
<srw:recordSchema>marcxchange</srw:recordSchema>
<srw:recordPacking>xml</srw:recordPacking>
<srw:recordData>
<mxc:record format="INTERMARC" id="ark:/12148/cb41193642z" type="

#### Querying for JDPL and adding the result to the rest of the metadata

In [47]:
bnf_jdpl_query_uri = marc_API_URI.format(query=f"ark:/12148/{BNFEN_API_MAPPING['jdpl']}")

resp_jdpl = requests.get(bnf_jdpl_query_uri, timeout=60)
jdpl_contents = BeautifulSoup(resp_jdpl.content, 'xml')

jdpl_contents

<?xml version="1.0" encoding="utf-8"?>
<srw:searchRetrieveResponse xmlns="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:ixm="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:mn="http://catalogue.bnf.fr/namespaces/motsnotices" xmlns:sd="http://www.loc.gov/zing/srw/diagnostic/" xmlns:srw="http://www.loc.gov/zing/srw/">
<srw:version>1.2</srw:version>
<srw:echoedSearchRetrieveRequest>
<srw:version>1.2</srw:version>
<srw:query>(bib.ark any "ark:/12148/cb39294634r")</srw:query>
</srw:echoedSearchRetrieveRequest>
<srw:numberOfRecords>1</srw:numberOfRecords>
<srw:records>
<srw:record>
<srw:recordSchema>marcxchange</srw:recordSchema>
<srw:recordPacking>xml</srw:recordPacking>
<srw:recordData>
<mxc:record format="INTERMARC" id="ark:/12148/cb39294634r" type="Bibliographic" xmlns:mxc="info:lc/xmlns/marcxchange-v2"> <mxc:leader>02619c01s 2200027  345a </mxc:leader> <mxc:controlfield tag="001">FRBNF392946340000007</mxc:controlfield> <mxc:controlfield tag="003">http://catalogue.bnf.fr/ark:

In [48]:
# add the srw:record of JDPL to the other ones
all_contents.records.append(jdpl_contents.find('srw:record'))
all_contents

<?xml version="1.0" encoding="utf-8"?>
<srw:searchRetrieveResponse xmlns="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:ixm="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:mn="http://catalogue.bnf.fr/namespaces/motsnotices" xmlns:sd="http://www.loc.gov/zing/srw/diagnostic/" xmlns:srw="http://www.loc.gov/zing/srw/">
<srw:version>1.2</srw:version>
<srw:echoedSearchRetrieveRequest>
<srw:version>1.2</srw:version>
<srw:query>(bib.ark any "ark:/12148/cb32830550k ark:/12148/cb41193642z ark:/12148/cb328123058 ark:/12148/cb32836564q ark:/12148/cb39294634r ark:/12148/cb32779904b ark:/12148/cb34419111x ark:/12148/cb32771891w ark:/12148/cb327788531 ark:/12148/cb343488519 ark:/12148/cb34429265b")</srw:query>
</srw:echoedSearchRetrieveRequest>
<srw:numberOfRecords>11</srw:numberOfRecords>
<srw:records>
<srw:record>
<srw:recordSchema>marcxchange</srw:recordSchema>
<srw:recordPacking>xml</srw:recordPacking>
<srw:recordData>
<mxc:record format="INTERMARC" id="ark:/12148/cb41193642z" type="

#### Save the result to disk

In [49]:
with open(bnf_marc_file, mode="w") as file:
    file.write(str(all_contents))

#### Sanity check that all was correctly added and that there is indeed JPDL in the bnf stored data

In [50]:

bnf_recs = [r for r in pymarc.parse_xml_to_array(bnf_marc_file) if r is not None]
bnf_recs, len(bnf_recs)

([<pymarc.record.Record at 0x10e388e00>,
 11)

In [51]:
bnf_recs[-1].as_dict()

{'leader': '02619c01s 2200027  345a ',
 'fields': [{'001': 'FRBNF392946340000007'},
  {'003': 'http://catalogue.bnf.fr/ark:/12148/cb39294634r'},
  {'008': '041216d 1814 1944            frfre pd 7b      '},
  {'009': 'a                  '},
  {'022': {'ind1': '1',
    'ind2': ' ',
    'subfields': [{'a': '1770-619X'}, {'c': '1770-619X'}]}},
  {'051': {'ind1': ' ', 'ind2': ' ', 'subfields': [{'a': 'txt'}, {'b': 'n'}]}},
  {'222': {'ind1': '0',
    'ind2': ' ',
    'subfields': [{'a': 'Journal des débats politiques et littéraires'}]}},
  {'245': {'ind1': '1',
    'ind2': ' ',
    'subfields': [{'a': 'Journal des débats politiques et littéraires'},
     {'d': 'Texte imprimé'}]}},
  {'255': {'ind1': '2',
    'ind2': ' ',
    'subfields': [{'a': '1er avr. 1814'}, {'b': '19/20 août 1944'}]}},
  {'260': {'ind1': ' ',
    'ind2': ' ',
    'subfields': [{'a': 'Paris'}, {'c': '[s.n.]'}, {'d': '1814-1944'}]}},
  {'280': {'ind1': ' ',
    'ind2': ' ',
    'subfields': [{'d': 'In-fol. puis gr. fol.'